In [1]:
using DataFrames, CSV

"""
    load_baci(folder, years; product=270900, version="V202601", hs="HS17")

Loads the BACI files for the requested `years`, keeps only `product`
(default 270900, crude oil), and returns one DataFrame with named columns.
"""

function load_baci(folder::AbstractString, years::AbstractVector{<:Integer};
                   product::Integer = 270900,
                   version::AbstractString = "V202601",
                   hs::AbstractString = "HS17")

    # Country names
    cc = CSV.read(joinpath(folder, "country_codes_$(version).csv"), DataFrame;
                  select = ["country_code", "country_name"])
    cc.country_code = Int16.(cc.country_code)

    frames = DataFrame[]
    for y in years
        path = joinpath(folder, "BACI_$(hs)_Y$(y)_$(version).csv")
        isfile(path) || error("File not found: $path")

        df = CSV.read(path, DataFrame;
                      select = ["t", "i", "j", "k", "v", "q"],
                      types = Dict("t" => Int16, "i" => Int16, "j" => Int16,
                                   "k" => Int32, "v" => Float64, "q" => Float64),
                      missingstring = ["NA", ""])

        push!(frames, df[df.k .== product, :])      # keep only crude oil
    end

    out = reduce(vcat, frames)

    rename!(out, "t" => "year",
                 "i" => "exporter_code",
                 "j" => "importer_code",
                 "k" => "product",
                 "v" => "value_kUSD",
                 "q" => "quantity_tons")

    out = leftjoin(out, rename(cc, "country_code" => "exporter_code",
                                   "country_name" => "exporter"), on = "exporter_code")
    out = leftjoin(out, rename(cc, "country_code" => "importer_code",
                                   "country_name" => "importer"), on = "importer_code")

    select!(out, "year", "exporter_code", "exporter", "importer_code", "importer",
                 "product", "value_kUSD", "quantity_tons")
    sort!(out, ["year", "value_kUSD"], rev = [false, true])
    return out
end



load_baci (generic function with 1 method)

In [4]:
# Paste the folder path from the File Explorer address bar
# raw"..." keeps Windows backslashes as they are

RAW_FOLDER = raw"C:\Users\PC\OneDrive\UCF_Master\CECS26-27\CAP6318_AnalysisOfSocialComplexity\Project\BACI_HS17_V202601"

isdir(RAW_FOLDER)      # should be true
readdir(RAW_FOLDER)    # should list the BACI_HS17_... files

7-element Vector{String}:
 "BACI_HS17_Y2021_V202601.csv"
 "BACI_HS17_Y2022_V202601.csv"
 "BACI_HS17_Y2023_V202601.csv"
 "BACI_HS17_Y2024_V202601.csv"
 "Readme.txt"
 "country_codes_V202601.csv"
 "product_codes_HS17_V202601.csv"

In [16]:
@time crude21 = load_baci(RAW_FOLDER, [2021])
println(size(crude21))
first(crude21, 10)

  2.568978 seconds (24.31 k allocations: 1.174 GiB, 27.33% gc time)
(1515, 8)


Row,year,exporter_code,exporter,importer_code,importer,product,value_kUSD,quantity_tons
,Int16,Int16,DataStri…?,Int16,DataStri…?,Int32,Float64,Float64?
1,2021,124,Canada,842,USA,270900,7.98477e7,1.93903e8
2,2021,682,Saudi Arabia,156,China,270900,4.39386e7,8.75572e7
3,2021,643,Russian Federation,156,China,270900,3.54466e7,7.29495e7
4,2021,368,Iraq,156,China,270900,2.64716e7,5.40692e7
5,2021,682,Saudi Arabia,392,Japan,270900,2.52248e7,4.86788e7
6,2021,368,Iraq,699,India,270900,2.50522e7,5.18827e7
7,2021,512,Oman,156,China,270900,2.24616e7,4.48146e7
8,2021,784,United Arab Emirates,392,Japan,270900,2.19487e7,4.159e7
9,2021,682,Saudi Arabia,410,Rep. of Korea,270900,2.13205e7,4.05145e7


In [17]:
YEARS = 2021:2024

crude = Dict{Int,DataFrame}()          # cleaned data per year, e.g. crude[2024]
check = DataFrame(year = Int[], rows = Int[],
                  missing_exporter = Int[], missing_importer = Int[],
                  missing_quantity = Int[], missing_value = Int[],
                  pct_missing_quantity = Float64[])

for y in YEARS
    file = joinpath(RAW_FOLDER, "BACI_HS17_Y$(y)_V202601.csv")
    if !isfile(file)
        println("Skipped $y (file not found)")
        continue
    end

    df = load_baci(RAW_FOLDER, [y])
    crude[y] = df

    n  = nrow(df)
    mq = count(ismissing, df.quantity_tons)
    push!(check, (y, n,
                  count(ismissing, df.exporter),
                  count(ismissing, df.importer),
                  mq,
                  count(ismissing, df.value_kUSD),
                  round(100 * mq / n, digits = 2)))
    println("Done $y")
end

check

Done 2021
Done 2022
Done 2023
Done 2024


Row,year,rows,missing_exporter,missing_importer,missing_quantity,missing_value,pct_missing_quantity
,Int64,Int64,Int64,Int64,Int64,Int64,Float64
1,2021,1515,0,0,46,0,3.04
2,2022,1521,0,0,54,0,3.55
3,2023,1435,0,0,45,0,3.14
4,2024,1260,0,0,47,0,3.73


In [ ]:
## Compare with clean_data.zip

ZIP_FOLDER = raw"C:\Users\PC\Downloads\clean_data\clean_data"   # unzipped clean files
old = CSV.read(joinpath(ZIP_FOLDER, "crude_2021.csv"), DataFrame)

# Sort both tables the same way before comparing cell by cell
keys_ = [:year, :exporter_code, :importer_code]
a = sort(crude21, keys_)
b = sort(old, keys_)

println("Rows (new) = ", nrow(a), " | Rows (zip) = ", nrow(b))
println("Same columns: ", names(a) == names(b))
if nrow(a) == nrow(b)
    for c in names(a)
        println(rpad(c, 15), all(isequal.(a[!, c], b[!, c])) ? "✓ match" : "✗ differ")
    end
end

Rows (new) = 1515 | Rows (zip) = 1515
Same columns: true
year           ✓ match
exporter_code  ✓ match
exporter       ✓ match
importer_code  ✓ match
importer       ✓ match
product        ✓ match
value_kUSD     ✓ match
quantity_tons  ✓ match


In [22]:
OUT_FOLDER = "data/clean_data_julia"
mkpath(OUT_FOLDER)

keys_ = [:year, :exporter_code, :importer_code]
result = DataFrame(year = Int[], rows_new = Int[], rows_zip = Union{Int,Missing}[],
                   saved = Bool[], all_match = Union{Bool,Missing}[])

for y in 2021:2024
    if !isfile(joinpath(RAW_FOLDER, "BACI_HS17_Y$(y)_V202601.csv"))
        println("Skipped $y (raw file not found)")
        continue
    end

    # Reuse the year if it was already loaded in the summary cell
    c = haskey(crude, y) ? crude[y] : load_baci(RAW_FOLDER, [y])
    CSV.write(joinpath(OUT_FOLDER, "crude_$(y).csv"), c)

    # Compare with the file from clean_data.zip
    zf = joinpath(ZIP_FOLDER, "crude_$(y).csv")
    if isfile(zf)
        a = sort(c, keys_)
        b = sort(CSV.read(zf, DataFrame), keys_)
        ok = nrow(a) == nrow(b) && names(a) == names(b) &&
             all(all(isequal.(a[!, col], b[!, col])) for col in names(a))
        push!(result, (y, nrow(a), nrow(b), true, ok))
    else
        push!(result, (y, nrow(c), missing, true, missing))
    end
end

result

Row,year,rows_new,rows_zip,saved,all_match
,Int64,Int64,Int64?,Bool,Bool?
1,2021,1515,1515,true,true
2,2022,1521,1521,true,true
3,2023,1435,1435,true,true
4,2024,1260,1260,true,true
